## Importing modules

In [ ]:
import numpy as np
import h5py
import tensorly as tl
import pickle

from vib_frags import tensor_utils as tu
from vib_frags import cost_utils as cu
from vib_frags import cp3_utils as cpu

## Importing Hamiltonian

Hamiltonians used for Trotter estimates are smaller and are stored in a different convention from the PAH Hamiltonians. That distinction is made in how the loaded tensors are transposed.

In [ ]:
mol_name = 'pah1'                                                     #Can be 'pah1', 'pah2', ..., 'pah7', or 'H2S', 'CO2', 'CH2O', 'CH4' 
au_to_cm1 = 219474.6                                                  #Stored tensors are in atomic units
mc_level = 2                                                          #Mode-coupling level
nmodals = 5                                                           #Number of modals

datafile = h5py.File(rf'Hamiltonians/PAH_Hamiltonians/{mol_name}/{mol_name}_{mc_level}mode/{mol_name}_VSCF_{nmodals}M_{mc_level}MC.hdf5', 'r')

#Extracting Hamiltonian terms at different mode-coupling levels
#One-mode-coupling term
H1_full = np.array(datafile['H1'][()]) * au_to_cm1     
H1 = H1_full[:, :nmodals, :nmodals]

#Two-mode-coupling term
H2_full = np.array(datafile['H2'][()]) * au_to_cm1      
H2_nonsym = np.transpose(H2_full, (3,4,5,0,1,2))
H2_nonsym = H2_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals]       #Non symmetrized tensor
H2 = tu.symmetrize_tbt(H2_nonsym)

#Three-mode-coupling term
try:
  H3_full = np.array(datafile['H3'][()]) * au_to_cm1
  H3_nonsym = np.transpose(H3_full, (6,7,8,3,4,5,0,1,2))
  H3_nonsym = H3_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals, :, :nmodals, :nmodals]
  H3 = tu.symmetrize_trbt(H3_nonsym)
except KeyError:
  print ("3 mode term is missing. Setting 3 mode tensor to None.")
  H3_full, H3_nonsym, H3 = None, None, None

nmodes = H1.shape[0]
print ('Number of modes:', nmodes)
print ('Number of modals:', nmodals)

## Triangular LCU

### For three-mode-coupling term

There are some hyperparameters in the optimization below that needs to be manually vaaried to obtain the desired tensor approximation error. These hyperparameters enter through the optional arguments of Tensorly's CP-ALS function `tl.decomposition.parafac`. Running this cell also requires precomputed THC fragments which we provide for the case of PAH1 and PAH2 Hamiltonians. Specifically, we provide the THC parameters obtained through CP3 based initialization as it can comfortably reach the desired target accuracy when number of modals is small as in the PAH1 and PAH2 three-mode-coupling Hamiltonians.

In [ ]:
#Loading the precomputed CP3 parameters that represent a THC factorization of three-mode-coupling Hamiltonian
with open (f"Hamiltonians/PAH_Hamiltonians/{mol_name}/{mol_name}_{mc_level}mode/CP3_trbt_params_{nmodals}M_hR_{3}MC_eps_1.pkl", 'rb') as in_params:
  print (f"Loading from: Hamiltonians/PAH_Hamiltonians/{mol_name}/{mol_name}_{mc_level}mode/CP3_trbt_params_{nmodals}M_hR_{3}MC_eps_1.pkl")
  _hcp3_params_ld = pickle.load(in_params)

thetas_dict = _hcp3_params_ld['theta']
gammas_dict = _hcp3_params_ld['gamma']
H3_nonsym_approx = np.zeros_like(H3_nonsym)
num_frags = {}
trbt_one_norm = 0

tl.set_backend('numpy')
np.random.seed(42)

rr, cc = np.triu_indices(nmodals)

#The \chi factors of THC
Xs_dict = {}
for i in range (nmodes):
  Xs_dict[i] = cpu.angles_to_unit_vectors(thetas_dict[i])

#For each tuple (i,j,k), obtain a sum of products of three single-mode-coupling operators acting on modes i,j, and k
for i in range (nmodes):
  XipqU = np.einsum('pU,qU->Upq', Xs_dict[i], Xs_dict[i], optimize = True)
  for j in range (i):
    XjrsV = np.einsum('rV,sV->Vrs', Xs_dict[j], Xs_dict[j], optimize = True)
    for k in range (j):
      XktuW = np.einsum('tW,uW->Wtu', Xs_dict[k], Xs_dict[k], optimize = True)
      gamma_ijk = gammas_dict[(i,j,k)]
      init_rank = 1
      init_guess = 'svd'
      diff_err = 1

      #Perform CP-ALS with increasing rank until rank = 7. This can be varied but we foudn that rank=7 cutoff sufficed for the two PAH molecules considered here.
      while diff_err > 1e-3:
        (weights, factors), errs = tl.decomposition.parafac(gamma_ijk, rank=init_rank, tol=1e-1, normalize_factors=False, orthogonalise=False, linesearch=True, init=init_guess, return_errors = True, n_iter_max=5)
        weights_rscld = [np.cbrt(wt) for wt in weights]
        aipq_term = np.einsum('a, Ua, Upq -> apq', weights_rscld, factors[0], XipqU, optimize=True)
        ajrs_term = np.einsum('a, Va, Vrs -> ars', weights_rscld, factors[1], XjrsV, optimize=True)
        aktu_term = np.einsum('a, Wa, Wtu -> atu', weights_rscld, factors[2], XktuW, optimize=True)
        ijk_approx = np.einsum('apq, ars, atu -> pqrstu', aipq_term, ajrs_term, aktu_term, optimize=True)
        diff_err = np.sqrt(np.sum((ijk_approx - H3_nonsym[i,:,:,j,:,:,k,:,:])**2))
        
        #Use the factors obtained in the previous round as an initial guess for the next round
        init_guess = (np.hstack((weights, [1e-5])), [np.hstack((factors[i], np.random.uniform(size=(gamma_ijk.shape[i],1)))) for i in range (3)])
        init_rank += 1
        if init_rank > 7:
          break

      H3_nonsym_approx[i,:,:,j,:,:,k,:,:] = ijk_approx

      # Slicing with [:, r, c] extracts only the upper triangle elements into a smaller 2D array.
      sum_a = np.sum(np.abs(aipq_term[:, rr, cc]), axis=1)
      sum_b = np.sum(np.abs(ajrs_term[:, rr, cc]), axis=1)
      sum_c = np.sum(np.abs(aktu_term[:, rr, cc]), axis=1)

      # Multiply and sum the resulting 1D arrays
      trbt_one_norm += np.sum(sum_a * sum_b * sum_c)

      num_frags[(i,j,k)] = len(weights)

      print ("Mode tuple:", (i,j,k), "Number of fragments:", len(weights), "Decomposition error:", diff_err, "cm^-1")
      
trbt_err = 0
for i in range (nmodes):
  for j in range (i):
    for k in range (j):
      exact_ijk = H3_nonsym[i,:,:,j,:,:,k,:,:]
      approx_ijk = H3_nonsym_approx[i,:,:,j,:,:,k,:,:]
      trbt_err += np.sum((exact_ijk - approx_ijk)**2)
trbt_err = np.sqrt(trbt_err)
print ("\nFinal tensor error: ", trbt_err, "cm-1")

print ("Trbt one-norm: ", trbt_one_norm, "cm-1")
trbt_QROM_vol = sum(list(num_frags.values()))*3*nmodals*(nmodals+1)/2
print ("Trbt QROM vol: ", trbt_QROM_vol)
num_trbt_frags = sum(list(num_frags.values()))

### For two-mode-coupling term

Use directly when the Hamiltonian has no three-mode-coupling term. Otherwise, run the above cell first to get the three-mode-coupling fragments.

In [ ]:
#Snigular value cutoffs used for the PAH Hamiltonians in the paper
cutoff_data = {'pah1': {2: 0.1392, 3: 0.08}, 'pah2': {2: 0.11, 3: 0.065}, 'pah3': {2: 0.0759}, 'pah4': {2: 0.06843}, 'pah5': {2: 0.0601}, 'pah6': {2: 0.0505}, 'pah7': {2: 0.04258}}

obt_one_norm = np.sum(np.abs(np.triu(H1)))
tbt_one_norm = 0
num_1mode_ops = nmodes                                    #Coming from the nmode number of one mode operators in the one-mode-coupling term of the Hamiltonian
H2_nonsym_approx = np.zeros_like(H2)

#QROM_vol of one-mode-coupling term
QROM_vol = nmodes*nmodals*(nmodals+1)/2

#Obtaining triangular LCU for two-mode-coupling term
sig_cutoff = cutoff_data[mol_name][mc_level]
for i in range (nmodes):
  for j in range (i):
    H2ij_nonsym = H2_nonsym[i, :, :, j, :, :]
    H2ij_mat = H2ij_nonsym.reshape((nmodals**2, nmodals**2))
    U, sig, Vh = np.linalg.svd(H2ij_mat)
    imp_sig_idx = np.where(sig > sig_cutoff)[0]
    sig = sig[imp_sig_idx]
    Uh, Vh = np.diag(np.sqrt(sig)) @ U.T[imp_sig_idx, :], np.diag(np.sqrt(sig)) @ Vh[imp_sig_idx, :]
    H2_nonsym_approx[i, :, :, j, :, :] = (Uh.T @ Vh).reshape(H2ij_nonsym.shape)
    #Counting the total number of one-mode operators as the final cost directly depends on this.
    num_1mode_ops += 2*len(sig)

    Uh_mat = Uh.reshape((len(imp_sig_idx), nmodals, nmodals))
    Vh_mat = Vh.reshape((len(imp_sig_idx), nmodals, nmodals))
    tri_Uh_mat = np.triu(Uh_mat)
    tri_Vh_mat = np.triu(Vh_mat)
    #Two-mode-coupling 1-norm for the mode-pair (i,j)
    tbt_one_norm += np.sum(np.sum(np.abs(tri_Uh_mat), axis=(1,2)) * np.sum(np.abs(tri_Vh_mat), axis=(1,2)))
    #QROM volume for the mode-pair (i,j)
    QROM_vol += len(np.where(np.abs(tri_Uh_mat.ravel()) > 1e-5)[0]) + len(np.where(np.abs(tri_Vh_mat.ravel()) > 1e-5)[0])

num_tbt_frags = int((num_1mode_ops - nmodes)/2)
num_tot_frags = nmodes + num_tbt_frags

tot_one_norm = obt_one_norm + tbt_one_norm
if mc_level == 3:
  tot_one_norm += trbt_one_norm
  QROM_vol += trbt_QROM_vol
  num_1mode_ops += sum(list(num_frags.values()))*3
  num_tot_frags += num_trbt_frags
print (f"Singular value cutoff used: ", sig_cutoff)
print ("Number of coefficients to load: ", QROM_vol)

### QPE cost

In [ ]:
#Cost of Triangular LCU according to Majland et al (with some additional necessary overheads)
aleph = 20                           #Bits of precision used to represent LCU coefficients
eps = 1                              #Target energy accuracy in cm-1
NN = int(nmodals*(nmodals+1)/2)      #Number of terms in the LCU
CP1 = NN + aleph - 2
CPdag1 = CP1
CS1 = NN + nmodals*(nmodals-1) - 1                                                        #From equation 53 and the factor of 2 is to ensure both Pauli X and Pauli Y operators are implemented
CBE1 = CP1 + CPdag1 + CS1


C_BE = num_1mode_ops*CBE1
CR = 2*num_tbt_frags*(aleph + np.ceil(np.log2(nmodals*(nmodals+1)/2)) + 1 - 2) #All reflections between one-body operators being alephltiplied to ensure a product is formed
if mc_level == 3:
  CR += 3*num_trbt_frags*(aleph + np.ceil(np.log2(nmodals*(nmodals+1)/2)) + 1 - 2)
C_BE += CR
QPE_nsteps = np.ceil(np.pi*tot_one_norm/(2*eps))
C_Tri = QPE_nsteps*C_BE
print (f"Cost of BE: {C_BE:0.2e}")
print (f"LCU 1-norm: {tot_one_norm:0.2e} cm-1")
print (f"Toffoli cost of Triangular LCU: {C_Tri:0.2e}")

QPE_unary_qubits = 2*np.ceil(np.log2(QPE_nsteps + 1)) - 1
first_PREP_qubits = 2*np.ceil(np.log2(num_tot_frags)) - 1 + 2 + 2*aleph + 1
second_PREP_SEL_qubits = (2*np.ceil(np.log2(nmodals*(nmodals+1)/2)) - 1 + 2 + 4*np.ceil(np.log2(nmodals)) + 2 + aleph + 1) + mc_level + nmodes*nmodals
N_Tri = int(QPE_unary_qubits + first_PREP_qubits + second_PREP_SEL_qubits)
print ("Qubit cost of Triangular LCU: ", N_Tri)


tbt_err = np.sqrt(np.sum((H2_nonsym - H2_nonsym_approx)**2))
if mc_level == 3:
  print ("\nError in two-mode-coupling tensor approximation = ", tbt_err, "cm-1")
  print ("Error in three-mode-coupling tensor approximation = ", trbt_err, "cm-1")
  print ("Total error in tensor approximation = ", tbt_err + trbt_err, "cm-1")
else:
  print ("\nTensor approximation error: ", tbt_err, "cm-1")
print ("Target accuracy in energy: ", 1, "cm-1")